# Auto-Tune Pipeline: Raw Data to Coupling Parameters

The `identify_binding_spec()` pipeline takes raw multichannel time series
and recovers coupling parameters in three stages:

1. **Hilbert phase extraction** -- analytic signal per channel gives
   instantaneous phase and dominant frequency via FFT peak.
2. **DMD frequency identification** -- SVD-based Dynamic Mode Decomposition
   extracts global modes and assigns channels to frequency clusters.
3. **Least-squares coupling estimation** -- fits
   `d\u03b8_i/dt - \u03c9_i = \u03a3_j K_ij sin(\u03b8_j - \u03b8_i)` via pseudoinverse
   to recover the coupling matrix K_ij from observed phase trajectories.

This notebook generates synthetic Kuramoto data with known coupling,
runs the pipeline, and compares estimated K_ij against ground truth.

In [ ]:
import numpy as np

from scpn_phase_orchestrator.autotune.pipeline import identify_binding_spec
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

In [ ]:
# Generate synthetic Kuramoto data: 4 oscillators with known coupling.
# Ground truth: K_ij = 0.5 * exp(-0.3 * |i-j|), zero diagonal.
N_OSC = 4
DT = 0.005
FS = 1.0 / DT  # 200 Hz
N_SAMPLES = 4000
TWO_PI = 2.0 * np.pi

# Known natural frequencies
true_omegas = np.array([1.0, 1.15, 0.9, 1.05])

# Known coupling matrix
idx = np.arange(N_OSC)
dist = np.abs(idx[:, np.newaxis] - idx[np.newaxis, :])
true_knm = 0.5 * np.exp(-0.3 * dist)
np.fill_diagonal(true_knm, 0.0)
true_alpha = np.zeros((N_OSC, N_OSC))

print("Ground truth K_ij:")
print(np.round(true_knm, 4))

# Simulate phase trajectories
engine = UPDEEngine(N_OSC, dt=DT, method="rk4")
rng = np.random.default_rng(42)
phases = rng.uniform(0, TWO_PI, N_OSC)
phase_traj = np.zeros((N_OSC, N_SAMPLES))

for t in range(N_SAMPLES):
    phase_traj[:, t] = phases
    phases = engine.step(phases, true_omegas, true_knm, 0.0, 0.0, true_alpha)

# Convert phase trajectories to synthetic "raw" signals: x_i(t) = cos(theta_i(t))
raw_signals = np.cos(phase_traj)
print(f"\nGenerated {N_SAMPLES} samples at {FS} Hz")
print(f"Signal shape: {raw_signals.shape}")

In [ ]:
# Run the auto-tune pipeline
result = identify_binding_spec(raw_signals, fs=FS)

print("Estimated natural frequencies (rad/s):")
for i, (est, true) in enumerate(zip(result.omegas, true_omegas, strict=False)):
    print(f"  osc {i}: est={est:.4f}  true={true:.4f}  err={abs(est - true):.4f}")

print(f"\nEstimated K_c: {result.K_c_estimate:.4f}")
print(f"n_layers: {result.n_layers}")

In [ ]:
# Compare estimated K_ij against ground truth
print("Estimated K_ij:")
print(np.round(result.knm, 4))
print("\nGround truth K_ij:")
print(np.round(true_knm, 4))

# Element-wise relative error (excluding diagonal)
mask = ~np.eye(N_OSC, dtype=bool)
abs_err = np.abs(result.knm[mask] - true_knm[mask])
rel_err = abs_err / (true_knm[mask] + 1e-10)
print(f"\nMean absolute error: {np.mean(abs_err):.4f}")
print(f"Mean relative error: {np.mean(rel_err):.2%}")
print(f"Max absolute error:  {np.max(abs_err):.4f}")

In [ ]:
# Simulate with estimated parameters, compare R convergence
STEPS = 2000
engine_est = UPDEEngine(N_OSC, dt=DT, method="rk4")
engine_true = UPDEEngine(N_OSC, dt=DT, method="rk4")

est_omegas = np.array(result.omegas)
est_knm = result.knm
est_alpha = result.alpha

# Same initial conditions
rng2 = np.random.default_rng(99)
init_phases = rng2.uniform(0, TWO_PI, N_OSC)
phases_est = init_phases.copy()
phases_true = init_phases.copy()

R_est_hist, R_true_hist = [], []
for _ in range(STEPS):
    phases_est = engine_est.step(phases_est, est_omegas, est_knm, 0.0, 0.0, est_alpha)
    phases_true = engine_true.step(
        phases_true, true_omegas, true_knm, 0.0, 0.0, true_alpha
    )
    R_e, _ = compute_order_parameter(phases_est)
    R_t, _ = compute_order_parameter(phases_true)
    R_est_hist.append(R_e)
    R_true_hist.append(R_t)

try:
    import matplotlib.pyplot as plt

    t = np.arange(STEPS) * DT
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, R_true_hist, label="True K_ij", linewidth=1.2)
    ax.plot(t, R_est_hist, label="Estimated K_ij", linewidth=1.2, linestyle="--")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("R (order parameter)")
    ax.set_title("R Convergence: True vs Estimated Parameters")
    ax.set_ylim(-0.05, 1.1)
    ax.legend()
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

print(f"\nFinal R (true params):      {R_true_hist[-1]:.4f}")
print(f"Final R (estimated params): {R_est_hist[-1]:.4f}")